# Lab 15-03 · Time Response and Specifications

Four parts, each built on `specs.py`. **Predict first**, then run.

In [1]:
import math
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from specs import step_specs, step_curve, second_order, zeta_from_overshoot

## Part 1 · The second-order table

*Predict:* is the exact 2% settling time a smooth function of $\zeta$?

In [2]:
zetas = np.linspace(0.2, 1.2, 201)
rows = [step_specs([1], [1, 2 * z, 1], 60, n=60001) for z in zetas]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.2))
a1.plot(zetas, [100 * r["overshoot"] for r in rows])
a1.set_xlabel("ζ"); a1.set_ylabel("overshoot (%)"); a1.grid(True, alpha=0.3)
a2.plot(zetas, [r["settle"] for r in rows], label="exact ωn·ts (2%)")
a2.plot(zetas, 4 / zetas, "--", label="4/ζ rule")
a2.plot(zetas, [r["rise"] for r in rows], label="exact ωn·tr")
a2.axhline(1.8, color="gray", ls=":", label="1.8 rule")
a2.set_xlabel("ζ"); a2.legend(fontsize=7); a2.grid(True, alpha=0.3)
plt.show()
for z in (0.3, 0.5, 1 / math.sqrt(2), 0.8, 1.0):
    r = step_specs([1], [1, 2 * z, 1], 60)
    print(f"zeta {z:.3f}: overshoot {100 * r['overshoot']:6.2f}%  wn*tr {r['rise']:.3f}  wn*ts {r['settle']:.3f}")

zeta 0.300: overshoot  37.23%  wn*tr 1.321  wn*ts 11.230


zeta 0.500: overshoot  16.30%  wn*tr 1.638  wn*ts 8.076


zeta 0.707: overshoot   4.32%  wn*tr 2.148  wn*ts 5.963


zeta 0.800: overshoot   1.52%  wn*tr 2.467  wn*ts 3.756


zeta 1.000: overshoot   0.00%  wn*tr 3.358  wn*ts 5.834


## Part 2 · An RLC circuit in ngspice

*Predict:* overshoot and peak time for $\zeta$ = 0.5, $\omega_n$ = 31,623 rad/s.

In [3]:
R, L, C = 31.6228, 1e-3, 1e-6
r = step_specs([1], [L * C, R * C, 1], 2e-3)
print(f"Python : rise {r['rise'] * 1e6:.2f} us, peak at {r['tpeak'] * 1e6:.2f} us, overshoot {100 * r['overshoot']:.2f}%, settle {r['settle'] * 1e6:.2f} us")
if shutil.which("ngspice"):
    out = subprocess.run(["ngspice", "-b", "rlc_step.cir"], capture_output=True, text=True, timeout=120).stdout
    print("ngspice:", "  ".join(l.strip() for l in out.splitlines() if l.split("=")[0].strip() in ("tr", "tp", "os", "ts") and l.count("=") == 1))
else:
    print("ngspice not installed: skip (CI runs rlc_step.cir through tools/run_spice.py)")

Python : rise 51.78 us, peak at 114.71 us, overshoot 16.30%, settle 255.40 us


ngspice: tp                  =  1.147153e-04  tr = 5.178462e-05  tp = 1.147153e-04  os = 1.630330e+01  ts = 2.553970e-04


## Part 3 · Extra poles, zeros and a doublet

In [4]:
base_den = [1, 1, 1]
print("extra pole p:")
for p in (None, 10, 3, 1):
    num, den = ([1], base_den) if p is None else ([p], np.polymul(base_den, [1, p]))
    r = step_specs(num, den, 40)
    print(f"  p = {str(p):4s}: overshoot {100 * r['overshoot']:5.1f}%  rise {r['rise']:.2f} s  settle {r['settle']:.2f} s")
print("zero:")
for z in (2, 0.5, -2):
    r = step_specs([1 / z, 1], base_den, 40)
    print(f"  z = {z:4}: overshoot {100 * r['overshoot']:5.1f}%  undershoot {100 * r['undershoot']:.2f}%  rise {r['rise']:.3f} s")

extra pole p:


  p = None: overshoot  16.3%  rise 1.64 s  settle 8.08 s


  p = 10  : overshoot  16.2%  rise 1.65 s  settle 8.17 s


  p = 3   : overshoot  15.2%  rise 1.76 s  settle 8.35 s


  p = 1   : overshoot   8.1%  rise 2.29 s  settle 6.64 s
zero:


  z =    2: overshoot  19.1%  undershoot 0.00%  rise 1.375 s


  z =  0.5: overshoot  69.9%  undershoot 0.00%  rise 0.479 s


  z =   -2: overshoot  17.8%  undershoot 9.12%  rise 1.478 s


In [5]:
wc, p, z = 2 * math.pi * 100e6, 2 * math.pi * 1e6, 2 * math.pi * 1.05e6
num = np.polymul([wc], [1, z]) * p / z
den = np.polymul([1, wc], [1, p])
t = np.linspace(1e-10, 1.2e-6, 6001)
err = np.abs(1 - step_curve(num, den, t))
err0 = np.abs(1 - step_curve([wc], [1, wc], t))
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.semilogy(t * 1e9, err, label="with doublet")
ax.semilogy(t * 1e9, np.maximum(err0, 1e-16), label="without")
ax.axhline(1e-3, color="gray", ls=":", label="0.1%")
ax.set_ylim(1e-6, 1.5); ax.set_xlabel("t (ns)"); ax.set_ylabel("|error|"); ax.legend(); ax.grid(True, alpha=0.3)
plt.show()
r = step_specs(num, den, 2e-6, band=1e-3, n=400001)
print(f"0.1% settling with the doublet: {r['settle'] * 1e9:.0f} ns")

0.1% settling with the doublet: 616 ns


## Part 4 · Specifications to poles

Overshoot ≤ 5%, 2% settling ≤ 2 s, rise time ≤ 0.6 s.

In [6]:
zmin = zeta_from_overshoot(0.05)
print(f"rules: zeta >= {zmin:.3f}, sigma >= {4 / 2:.1f}, wn >= {1.8 / 0.6:.1f}")
for name, pole in (("A", -2 + 2j), ("B", -3 + 2j), ("C", -1 + 1j), ("D", -4 + 5j), ("E", -4 + 3j)):
    wn, sig = abs(pole), -pole.real
    rules = (sig / wn >= zmin) and (sig >= 2) and (wn >= 3)
    r = step_specs([wn**2], [1, 2 * sig, wn**2], 20)
    exact = (r["overshoot"] <= 0.05) and (r["settle"] <= 2) and (r["rise"] <= 0.6)
    print(f"{name} {pole}: rules {'pass' if rules else 'fail'}, exact {'pass' if exact else 'fail'} "
          f"(OS {100 * r['overshoot']:.2f}%, ts {r['settle']:.2f} s, tr {r['rise']:.3f} s)")

rules: zeta >= 0.690, sigma >= 2.0, wn >= 3.0


A (-2+2j): rules fail, exact fail (OS 4.32%, ts 2.11 s, tr 0.759 s)


B (-3+2j): rules pass, exact fail (OS 0.90%, ts 1.12 s, tr 0.719 s)


C (-1+1j): rules fail, exact fail (OS 4.32%, ts 4.22 s, tr 1.519 s)


D (-4+5j): rules fail, exact fail (OS 8.10%, ts 0.93 s, tr 0.299 s)


E (-4+3j): rules pass, exact pass (OS 1.52%, ts 0.75 s, tr 0.493 s)


**Discussion.** Answer the four questions in the [lab handout](README.md) before reading its answers.